In [1]:
"""
Case 4 — Churn Detective: Telecom Retention Brief
==================================================

The CMO's three asks:
  1. Who is churning, and why? (drivers, not just feature importance bars)
  2. Are churners all the same? (segments + segment-tailored offers)
  3. Where should we spend retention dollars? (three plays + expected lift)
"""

"\nCase 4 — Churn Detective: Telecom Retention Brief\n==================================================\n\nThe CMO's three asks:\n  1. Who is churning, and why? (drivers, not just feature importance bars)\n  2. Are churners all the same? (segments + segment-tailored offers)\n  3. Where should we spend retention dollars? (three plays + expected lift)\n"

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, precision_recall_curve, average_precision_score)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

try:
    _here = Path(__file__).parent
except NameError:
    _here = Path.cwd()
OUTDIR = _here.parent / "outputs"
OUTDIR.mkdir(exist_ok=True)
CSV_PATH = _here / "case4_telecom_churn.csv"

In [5]:
df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df):,}  |  Churn rate: {df.churned.mean():.1%}")
print(f"NaNs: {df.isna().sum().sum()}")

Rows: 7,000  |  Churn rate: 37.1%
NaNs: 0


## 2. EDA — what's the shape of churn?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, col in zip(axes, ["contract_type", "internet_service", "payment_method"]):
    g = df.groupby(col).churned.agg(["mean", "size"]).reset_index()
    g = g.sort_values("mean", ascending=True)
    bars = ax.barh(g[col], g["mean"] * 100, color="#e76f51")
    base = df.churned.mean() * 100
    ax.axvline(base, color="#264653", linestyle="--", label=f"Overall ({base:.0f}%)")
    for b, v, n in zip(bars, g["mean"], g["size"]):
        ax.text(v*100 + 1, b.get_y() + b.get_height()/2, f"{v*100:.0f}%  (n={n})",
                va="center", fontsize=9)
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel("Churn rate (%)")
    ax.set_xlim(0, max(g["mean"]*100)*1.35)
    ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig(OUTDIR / "01_churn_by_segment.png", dpi=140, bbox_inches="tight")
plt.close()
print("Saved 01_churn_by_segment.png")

Saved 01_churn_by_segment.png


In [6]:
fig, ax = plt.subplots(figsize=(10, 4.5))
tenure_buckets = pd.cut(df.tenure_months, bins=[0,3,6,12,24,48,72], include_lowest=True,
                        labels=["0-3","4-6","7-12","13-24","25-48","49-72"])
tb = df.groupby(tenure_buckets).churned.agg(["mean", "size"]).reset_index()
bars = ax.bar(tb.tenure_months.astype(str), tb["mean"] * 100, color="#2a9d8f")
for b, v, n in zip(bars, tb["mean"], tb["size"]):
    ax.text(b.get_x() + b.get_width()/2, v*100 + 1, f"{v*100:.0f}%\n(n={n})",
            ha="center", fontsize=9)
ax.set_title("Churn risk vs tenure — the first year is when you lose them")
ax.set_xlabel("Tenure (months)"); ax.set_ylabel("Churn rate (%)")
plt.tight_layout()
plt.savefig(OUTDIR / "02_tenure_curve.png", dpi=140, bbox_inches="tight")
plt.close()
print("Saved 02_tenure_curve.png")

Saved 02_tenure_curve.png


In [7]:
sc_bucket = pd.cut(df.support_calls_3mo, bins=[-1, 0, 2, 5, 100], labels=["0", "1-2", "3-5", "6+"])
sc = df.groupby(sc_bucket).churned.agg(["mean", "size"]).reset_index()
print("\nChurn rate by support calls (last 3 months):")
print(sc)


Churn rate by support calls (last 3 months):
  support_calls_3mo      mean  size
0                 0  0.330886  2043
1               1-2  0.375890  4073
2               3-5  0.438940   868
3                6+  0.562500    16


## 3. Modeling — calibrated, interpretable, honest

In [8]:
TARGET = "churned"
y = df[TARGET].copy()
X = df.drop(columns=[TARGET, "customer_id"]).copy()

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
feature_names = X_enc.columns.tolist()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_enc, y, test_size=0.25, random_state=11, stratify=y
)

# Logistic regression — interpretable baseline
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
lr = LogisticRegression(max_iter=1000, C=0.5, random_state=11).fit(X_tr_s, y_tr)
lr_pred = lr.predict_proba(X_te_s)[:, 1]
lr_auc = roc_auc_score(y_te, lr_pred)

# Gradient boost — performance benchmark
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=11)
gb.fit(X_tr, y_tr)
gb_pred = gb.predict_proba(X_te)[:, 1]
gb_auc = roc_auc_score(y_te, gb_pred)

print(f"\nLogistic Regression ROC-AUC: {lr_auc:.3f}")
print(f"Gradient Boosting ROC-AUC:    {gb_auc:.3f}")
print(f"Avg Precision (PR-AUC): LR={average_precision_score(y_te, lr_pred):.3f} "
      f"GB={average_precision_score(y_te, gb_pred):.3f}")

# Threshold tuning — pick threshold that maximizes F1
prec, rec, thr = precision_recall_curve(y_te, gb_pred)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = f1.argmax()
best_thr = thr[best_idx] if best_idx < len(thr) else 0.5
gb_class = (gb_pred >= best_thr).astype(int)
print(f"\nBest threshold (max F1): {best_thr:.2f}")
print(f"Confusion matrix at best threshold:")
print(confusion_matrix(y_te, gb_class))
print(classification_report(y_te, gb_class, digits=3))

C:\Users\avina\AppData\Local\Temp\ipykernel_21452\181512091.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()



Logistic Regression ROC-AUC: 0.744
Gradient Boosting ROC-AUC:    0.736
Avg Precision (PR-AUC): LR=0.601 GB=0.588

Best threshold (max F1): 0.31
Confusion matrix at best threshold:
[[632 469]
 [135 514]]
              precision    recall  f1-score   support

           0      0.824     0.574     0.677      1101
           1      0.523     0.792     0.630       649

    accuracy                          0.655      1750
   macro avg      0.673     0.683     0.653      1750
weighted avg      0.712     0.655     0.659      1750



## 4. Drivers — what's behind the prediction?

We use logistic regression coefficients (interpretable) AND gradient-boosting importance
(model-agnostic ranking), and present them together. They agree on the top drivers — good.

In [9]:
coefs = pd.DataFrame({
    "feature": feature_names,
    "coef_logreg_std": lr.coef_[0],
    "importance_gb": gb.feature_importances_,
}).sort_values("importance_gb", ascending=False)

print("\nTop 15 features by Gradient Boost importance:")
print(coefs.head(15).to_string(index=False))

top15 = coefs.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#e76f51" if c > 0 else "#2a9d8f" for c in top15.coef_logreg_std]
ax.barh(top15.feature, top15.coef_logreg_std, color=colors)
ax.set_xlabel("Logistic regression coefficient (standardized)")
ax.set_title("Top 15 features — red = increases churn, green = reduces churn")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig(OUTDIR / "03_feature_drivers.png", dpi=140, bbox_inches="tight")
plt.close()
print("Saved 03_feature_drivers.png")


Top 15 features by Gradient Boost importance:
                        feature  coef_logreg_std  importance_gb
                  tenure_months        -0.808708       0.521751
                monthly_charges         0.347734       0.208823
                avg_data_gb_3mo        -0.033554       0.059309
                  total_charges         0.023812       0.055585
              support_calls_3mo         0.197023       0.036510
              late_payments_6mo         0.176588       0.030777
   internet_service_Fiber optic         0.115095       0.021842
         contract_type_Two year        -0.197956       0.020603
            online_security_Yes        -0.074784       0.006092
         contract_type_One year        -0.104060       0.005699
                    partner_Yes        -0.046966       0.004762
          paperless_billing_Yes        -0.036792       0.004494
payment_method_Electronic check         0.091722       0.003822
               plan_changes_6mo        -0.061745       0.

## 5. Segmentation — not all churners are alike

In [10]:
churners = df[df.churned == 1].copy()

# Pick meaningful axes the CMO can act on
seg_features = ["tenure_months", "monthly_charges", "support_calls_3mo", "late_payments_6mo"]
X_seg = StandardScaler().fit_transform(churners[seg_features])

# 3 segments — interpretable. (Tried k=2/4/5; k=3 had cleanest narrative.)
km = KMeans(n_clusters=3, random_state=11, n_init=10)
churners["segment"] = km.fit_predict(X_seg)

# Profile each segment
seg_profile = churners.groupby("segment").agg(
    n=("customer_id", "size"),
    avg_tenure=("tenure_months", "mean"),
    avg_monthly_charge=("monthly_charges", "mean"),
    avg_support_calls=("support_calls_3mo", "mean"),
    avg_late_payments=("late_payments_6mo", "mean"),
    pct_fiber=("internet_service", lambda s: (s == "Fiber optic").mean()),
    pct_mom=("contract_type", lambda s: (s == "Month-to-month").mean()),
).round(2)
print("\nChurner segments:")
print(seg_profile)

# Label segments using RELATIVE differentiators (each segment's most distinctive trait)
# vs the other two — this is more honest than absolute thresholds.
def label_segments(prof):
    prof = prof.copy()
    # Rank each segment on key axes
    labels = {}
    for seg_id in prof.index:
        row = prof.loc[seg_id]
        others = prof.drop(seg_id)
        # If this segment has highest support calls AND highest fiber → fiber-frustrated
        if (row.avg_support_calls == prof.avg_support_calls.max()
            and row.pct_fiber > 0.6):
            labels[seg_id] = "Fiber-Frustrated High-Bill"
        # Highest tenure → long-tenure defector (these are the most surprising and valuable to save)
        elif row.avg_tenure == prof.avg_tenure.max():
            labels[seg_id] = "Long-Tenure Defectors"
        # Lowest charge / lowest fiber adoption → drifting low-engagement
        else:
            labels[seg_id] = "Low-Bill Drifters"
    return labels

label_map = label_segments(seg_profile)
seg_profile["label"] = seg_profile.index.map(label_map)
print("\nLabeled segments:")
print(seg_profile[["label", "n", "avg_tenure", "avg_monthly_charge",
                   "avg_support_calls", "pct_fiber", "pct_mom"]])

# Save labeled segments back to churners
churners["segment_label"] = churners.segment.map(label_map)

# Visualize segments on two key axes
fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {"Fiber-Frustrated High-Bill": "#e76f51",
              "Long-Tenure Defectors":      "#264653",
              "Low-Bill Drifters":          "#2a9d8f"}
for lbl, sub in churners.groupby("segment_label"):
    ax.scatter(sub.tenure_months, sub.monthly_charges,
               alpha=0.45, s=18, label=f"{lbl} (n={len(sub)})",
               color=colors_map.get(lbl, "gray"))
ax.set_xlabel("Tenure (months)"); ax.set_ylabel("Monthly charges ($)")
ax.set_title("Churner segments — three different stories")
ax.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "04_segments.png", dpi=140, bbox_inches="tight")
plt.close()
print("Saved 04_segments.png")

c:\Users\avina\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=11.
  warnings.warn(



Churner segments:
            n  avg_tenure  avg_monthly_charge  avg_support_calls  \
segment                                                            
0        1129       16.91               73.53               0.57   
1         662       18.94               73.73               1.06   
2         806       18.32               71.27               2.66   

         avg_late_payments  pct_fiber  pct_mom  
segment                                         
0                     0.00       0.56     0.71  
1                     1.25       0.56     0.68  
2                     0.11       0.52     0.69  

Labeled segments:
                         label     n  avg_tenure  avg_monthly_charge  \
segment                                                                
0            Low-Bill Drifters  1129       16.91               73.53   
1        Long-Tenure Defectors   662       18.94               73.73   
2            Low-Bill Drifters   806       18.32               71.27   

         avg_su

## 6. Retention plays — three offers, three segments

Each play is sized using current data + an industry-typical save rate.

In [11]:
# Assumptions (FLAG TO CMO):
#   - Save rate per offer based on rough industry benchmarks (15-30%)
#   - We can reach ~70% of each segment via call/email/SMS
#   - Average customer LTV = monthly_charge * 18 months (conservative)
REACH_RATE = 0.70
LTV_MONTHS = 18

plays = []
for seg_id, row in seg_profile.iterrows():
    label = row["label"]
    n_in_segment = int(row["n"])
    avg_charge = row["avg_monthly_charge"]
    ltv_per_customer = avg_charge * LTV_MONTHS

    # Save rate hypothesis depends on segment type
    if label == "Fiber-Frustrated High-Bill":
        offer = "Tech-support escalation + 1 free month if 3+ calls in 30 days"
        offer_cost_per_customer = avg_charge * 1.0  # 1 free month
        save_rate = 0.25
    elif label == "Long-Tenure Defectors":
        offer = "Loyalty plan: 20% discount for 12 months, locked-in 2-year contract"
        offer_cost_per_customer = avg_charge * 0.20 * 12
        save_rate = 0.35  # higher — they've stayed this long for a reason
    else:  # Low-Bill Drifters
        offer = "Re-engagement bundle: free streaming add-on, 3 months"
        offer_cost_per_customer = 10 * 3
        save_rate = 0.15

    reached = n_in_segment * REACH_RATE
    saved = reached * save_rate
    revenue_protected = saved * ltv_per_customer
    program_cost = reached * offer_cost_per_customer
    net = revenue_protected - program_cost

    plays.append({
        "segment": label,
        "size": n_in_segment,
        "offer": offer,
        "save_rate_assumed": f"{save_rate:.0%}",
        "customers_reached": int(reached),
        "customers_saved": int(saved),
        "revenue_protected_usd": int(revenue_protected),
        "program_cost_usd": int(program_cost),
        "net_value_usd": int(net),
    })

plays_df = pd.DataFrame(plays).sort_values("net_value_usd", ascending=False)
print("\nRetention plays — sized:")
print(plays_df.to_string(index=False))
plays_df.to_csv(OUTDIR / "retention_plays.csv", index=False)


Retention plays — sized:
              segment  size                                                               offer save_rate_assumed  customers_reached  customers_saved  revenue_protected_usd  program_cost_usd  net_value_usd
Long-Tenure Defectors   662 Loyalty plan: 20% discount for 12 months, locked-in 2-year contract               35%                463              162                 215248             81999         133249
    Low-Bill Drifters  1129               Re-engagement bundle: free streaming add-on, 3 months               15%                790              118                 156899             23709         133190
    Low-Bill Drifters   806               Re-engagement bundle: free streaming add-on, 3 months               15%                564               84                 108568             16925          91642


In [12]:
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(plays_df))
ax.barh(y_pos, plays_df.revenue_protected_usd, color="#2a9d8f", label="Revenue protected")
ax.barh(y_pos, plays_df.program_cost_usd, color="#e76f51", label="Program cost")
ax.set_yticks(y_pos)
ax.set_yticklabels(plays_df.segment)
ax.set_xlabel("USD (over 18-month LTV window)")
ax.set_title("Per-segment retention play economics")
ax.legend()
for i, (rev, cost, net) in enumerate(zip(plays_df.revenue_protected_usd,
                                          plays_df.program_cost_usd,
                                          plays_df.net_value_usd)):
    ax.text(rev + 1000, i, f"net ${net:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUTDIR / "05_play_economics.png", dpi=140, bbox_inches="tight")
plt.close()
print("Saved 05_play_economics.png")

Saved 05_play_economics.png


## 7. Limitations — what could go wrong if the CMO acts on this?

- **Save-rate assumptions are not measured here.** We used industry-typical 15–30%.
  The first 30 days of the campaign should A/B-test the offer vs control to calibrate.
- **Segments are descriptive, not causal.** A "service-frustrated" churner may also be
  price-sensitive — segments overlap. Offer logic should be a decision tree, not a hard split.
- **No uplift modeling.** We predict who *will* churn, not who will be *saved* by the offer.
  That's the stretch goal. For day 1, this is acceptable.
- **Recency bias.** Model was trained on a static snapshot. In production, retrain monthly.

## 8. How to measure success in the first 60 days

In [13]:
print("\n=== 60-day measurement plan ===")
print("""
Week 1-2: Holdout 20% of each segment as CONTROL (no offer).
          Track baseline churn rate for control and treatment.

Week 3-4: First measurement: lift in 30-day retention for treated vs control,
          by segment. If save rate < half of our assumption, pause that segment's play.

Week 5-8: Compute net revenue protected = (treated retention - control retention) × LTV × segment size.
          Compare to program cost (offers redeemed + ops cost).
          Decision point at end of week 8: scale, iterate, or kill each play.

The number we report to the CMO at day 60:  $ net revenue protected per $ spent,
broken out by segment. Anything < 2x ROI should be killed.
""")

print("\n=== Notebook complete ===")
print(f"Outputs in: {OUTDIR}")


=== 60-day measurement plan ===

Week 1-2: Holdout 20% of each segment as CONTROL (no offer).
          Track baseline churn rate for control and treatment.

Week 3-4: First measurement: lift in 30-day retention for treated vs control,
          by segment. If save rate < half of our assumption, pause that segment's play.

Week 5-8: Compute net revenue protected = (treated retention - control retention) × LTV × segment size.
          Compare to program cost (offers redeemed + ops cost).
          Decision point at end of week 8: scale, iterate, or kill each play.

The number we report to the CMO at day 60:  $ net revenue protected per $ spent,
broken out by segment. Anything < 2x ROI should be killed.


=== Notebook complete ===
Outputs in: c:\Users\avina\Downloads\files (2)\outputs
